In [1]:
# !pip install segment-anything
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from segment_anything import sam_model_registry, SamPredictor
from segment_anything import sam_model_registry, SamPredictor
from segment_anything import SamAutomaticMaskGenerator

In [2]:
#A LA TERMINAL
# pip install torch torchvision torchaudio numpy opencv-python matplotlib
# pip install git+https://github.com/facebookresearch/segment-anything.git

# # Download a model checkpoint
# wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth


In [2]:
sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"  # Options: vit_h, vit_l, vit_b

# Load the model
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to("cuda" if torch.cuda.is_available() else "cpu")

# Create the predictor
predictor = SamPredictor(sam)

: 

In [ ]:
def is_near_center(mask, threshold=0.3):
    """Check if the mask is near the center of the image."""
    x, y, w, h = mask['bbox']
    mask_center_x, mask_center_y = x + w / 2, y + h / 2
    img_center_x, img_center_y = img_width / 2, img_height / 2

    # Normalize distances
    norm_dist_x = abs(mask_center_x - img_center_x) / img_width
    norm_dist_y = abs(mask_center_y - img_center_y) / img_height

    return norm_dist_x < threshold and norm_dist_y < threshold  # Must be near center

# Keep only masks near the center


def segment_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_generator = SamAutomaticMaskGenerator(sam)
    masks = mask_generator.generate(image)

    blank_canvas = np.ones_like(image) * 255  # White background
    blank_canvas_big = np.ones_like(image) * 255  # White background
    blank_canvas_center = np.ones_like(image) * 255  # White background

    for mask in masks:
        segmentation = mask["segmentation"]
        # Generate a random color
        color = np.random.randint(0, 255, (1, 3), dtype=np.uint8)[0].tolist()
        # Convert binary mask to contours
        contours, _ = cv2.findContours(segmentation.astype(np.uint8), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        # Draw contours on the image
        cv2.drawContours(blank_canvas, contours, -1, color, 2)

    baby_candidates = [m for m in masks if is_near_center(m)]
    MIN_BABY_SIZE = 10000  
    filtered_masks = [m for m in masks if m['area'] > MIN_BABY_SIZE]  # Keep only large regions

    for mask in baby_candidates:
        segmentation = mask["segmentation"]
        color = np.random.randint(0, 255, (1, 3), dtype=np.uint8)[0].tolist()
        contours, _ = cv2.findContours(segmentation.astype(np.uint8), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(blank_canvas_big, contours, -1, color, 2)

    for mask in filtered_masks:
        segmentation = mask["segmentation"]
        color = np.random.randint(0, 255, (1, 3), dtype=np.uint8)[0].tolist()
        contours, _ = cv2.findContours(segmentation.astype(np.uint8), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(blank_canvas_center, contours, -1, color, 2)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # Left: Original Image
    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    # Right: Segmentation Contours
    axes[1].imshow(blank_canvas)
    axes[1].set_title("Segmentation Contours")
    axes[1].axis("off")

    axes[2].imshow(blank_canvas_big)
    axes[2].set_title("big segments")
    axes[2].axis("off")

    axes[3].imshow(blank_canvas_center)
    axes[3].set_title("centered segments")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

    return image

In [ ]:
# img = segment_image('C:/Users/propietari/sintesisII/newborns/HM20241202041133.jpeg')
img = segment_image('C:/Users/propietari/sintesisII/newborns/HM20240816061812.jpeg')

In [ ]:
img = segment_image('C:/Users/propietari/sintesisII/newborns/HM20241202040525.jpeg')


In [ ]:
img = segment_image('C:/Users/propietari/sintesisII/newborns/HM20240815043811.jpeg')
